# The runoff of the plausible runs, and which parameters matter

**Experimental.** This notebook is part of a research project: the tools and
the choices in it are not settled.

The previous notebook (`dynamic_equifinality.ipynb`) found the plausible
parameter sets of OGGM's workflow for Vernagtferner: many combinations of the
MB parameters, Glen's A and the initial state of the spinup reproduce the
geodetic MB, the RGI area and the IceBoost mean thickness. They agree on these
observations by construction. Do they agree on what we want to know from the
model, for example its runoff?

Here we:

1. rerun the plausible runs with OGGM's hydrological output (`run_with_hydro`);
2. compute a few runoff metrics for 2000-2019: the mean annual runoff, its
   seasonal cycle, and the share of glacier melt in the runoff;
3. look at the spread of these metrics, and at which parameters drive it,
   with a very simple sensitivity analysis.

This is the second half of a GLUE-style analysis: keep the plausible runs,
then look at the spread of the predictions and at its sources. The metrics
here are examples; others can be computed from the same output.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from oggm import cfg, utils, workflow

from oggm_adapbs import weighted_quantile
from oggm_adapbs.dynamic import (select_particles, dynamic_hydro_rerun,
                                 runoff_metrics, save_netcdf, RUNOFF_VARIABLES)

C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
GREY = '#999999'

## The plausible runs

We start from the AdaPBS run of the previous notebook (with the initial state
period), saved in `out/wide_priors/` (by the notebook, or by
`scripts/run_glacier.py`, which writes the same files). The glacier directory
is the same preprocessed directory as there.

In [ ]:
rgi_id = 'RGI60-11.00719'  # Vernagtferner
out_dir = 'out/wide_priors'
fpath = f'{out_dir}/{rgi_id}_adapbs.nc'
if not os.path.exists(fpath):
    raise FileNotFoundError(f'{fpath} not found: run dynamic_equifinality.ipynb first.')
with xr.open_dataset(fpath) as d:
    ds = d.load()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir('adapbs_dynamic_runoff', reset=True)
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2025.6/elev_bands/W5E5/per_glacier/')
gdir = workflow.init_glacier_directories([rgi_id], from_prepro_level=3,
                                         prepro_border=80,
                                         prepro_base_url=base_url)[0]

# The parameters, with melt_f and prcp_fac themselves rather than their log10
params = ['melt_f', 'prcp_fac', 'temp_bias', 'log10_glen_a_factor',
          'initial_state_temp_offset']
w = ds.weight.values
print(f'{ds.sizes["particle"]} AdaPBS runs, effective sample size {1 / np.sum(w**2):.0f}')

## Which runs to rerun?

AdaPBS kept all its runs (9500 here), with their weights. Most of them have a
negligible weight: they come from the first iterations, far from the
observations. The AdaPBS runs did not compute the hydrological output, because
it makes each run about 2.5 times slower (OGGM recomputes the monthly MB of the
glacier and of the area around it). It is cheaper to rerun only the plausible
runs afterwards: the rerun gives exactly the same glacier.

`select_particles` offers two choices:

- the runs which hold 99.9% of the weight (the default): they keep their
  weights, nothing is lost;
- a random draw with the weights (`n_resample`): equally weighted runs (a run
  drawn several times counts several times), which is convenient for some
  analyses, and cheaper, but adds sampling noise.

In [ ]:
idx, wsel = select_particles(ds)  # 99.9% of the weight
idx_r, w_r = select_particles(ds, n_resample=500, seed=0)
print(f'99.9% of the weight: {len(idx)} runs to rerun')
print(f'500 random draws: {len(idx_r)} different runs to rerun')

How is the weight spread over the runs? We sort the runs by weight and add
their weights up, largest first:

In [ ]:
ws = np.sort(np.nan_to_num(w))[::-1] / np.nansum(w)
cum = np.cumsum(ws)
neff = 1 / np.sum(ws**2)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.arange(1, len(ws) + 1), cum, color=C[0])
ax.axvline(neff, color=GREY, ls='--', label=f'effective sample size ({neff:.0f})')
for q, ls in [(0.99, ':'), (0.999, '-.')]:
    n = np.searchsorted(cum, q) + 1
    ax.axvline(n, color='k', ls=ls, lw=1, label=f'{q:.1%} of the weight: {n} runs')
ax.set_xscale('log')
ax.set_xlabel('Number of runs (largest weights first)')
ax.set_ylabel('Cumulative weight')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
print('Weight of the largest runs:',
      ', '.join(f'{n}: {cum[n - 1]:.0%}' for n in [10, 50, 150, 500, 1000]))

The 10 largest weights hold 15% of the total, the 150 largest (about the
effective sample size) 71%, the 500 largest 98%. Going from 99.9% to 99% of
the weight would save about a quarter of the reruns (550 instead of 761); a
random draw of 500 runs
(below) needs about 240. The tail is not always this short: in an earlier
version of these notebooks (linear priors, 19 iterations), 99.9% of the weight
needed 2677 of 9500 runs.

Here we rerun the runs which hold 99.9% of the weight (a few minutes on 6
processes). As in the previous notebook, the result is saved
(`out/wide_priors/<rgi_id>_hydro.nc`) and loaded the next time: delete the
file to run it again.

**The catchment.** OGGM's runoff is the melt and the liquid precipitation on
the glacier and on the ice-free area around it, over a fixed catchment. By
default the catchment is the largest glacier area of the run. Here, the
largest area depends on the initial state of each run (a large glacier in 1974
makes a larger catchment), which would make the runoff of the plausible runs
differ for a reason which has nothing to do with the glacier. We use the
glacier area of 2000 instead (`ref_area_yr=2000`). One detail: OGGM makes the
catchment of each elevation band the larger of the reference area and the
glacier area, so in years when a run's glacier is larger than in 2000 (often
before 2000), its catchment is slightly larger.

In [ ]:
fpath = f'{out_dir}/{rgi_id}_hydro.nc'
if os.path.exists(fpath):
    with xr.open_dataset(fpath) as d:
        dh = d.load()
    # Made from this AdaPBS run? (the AdaPBS run may have been redone since)
    if not np.allclose(dh.temp_bias, ds.temp_bias.sel(particle=dh.particle)):
        raise ValueError(f'{fpath} does not match the AdaPBS run: delete it.')
    print(f'Loaded from {fpath} (delete the file to run it again)')
else:
    t0 = time.time()
    dh = dynamic_hydro_rerun(gdir, ds, idx, wsel, ref_area_yr=2000, n_workers=6)
    dh.attrs['wall_time_s'] = time.time() - t0
    save_netcdf(dh, fpath)
print(f'{dh.sizes["particle"]} runs, run time {dh.attrs["wall_time_s"]:.0f} s, '
      f'{int((dh.status != "ok").sum())} failed')
# The rerun gives the same glacier as the AdaPBS run
dv = np.abs(dh.volume_km3 - ds.volume_km3.sel(particle=dh.particle).sel(time=dh.time))
print(f'Largest volume difference with the AdaPBS runs: {float(dv.max()):.2e} km3')
dh

## Runoff of the plausible runs

The annual runoff and its components (melt and liquid precipitation, on and
off the glacier), as weighted median and 5-95% range of the plausible runs:

In [ ]:
def band(ax, t, x, weights, color, label, fill=True):
    # Weighted median and 5-95% range over the runs (first axis)
    q = np.array([weighted_quantile(x[:, i], weights, [0.05, 0.5, 0.95])
                  for i in range(x.shape[1])])
    if fill:
        ax.fill_between(t, q[:, 0], q[:, 2], color=color, alpha=0.25, lw=0)
    ax.plot(t, q[:, 1], color=color, label=label)


t = dh.time.values[:-1]  # the last year has no MB
wh = dh.weight.values
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
band(axes[0], t, dh.runoff.values[:, :-1] * 1e-9, wh, 'k', 'total runoff')
names = {'melt_on_glacier': 'melt on glacier', 'melt_off_glacier': 'melt off glacier',
         'liq_prcp_on_glacier': 'rain on glacier', 'liq_prcp_off_glacier': 'rain off glacier'}
for (v, lab), c in zip(names.items(), C):
    band(axes[1], t, dh[v].values[:, :-1] * 1e-9, wh, c, lab)
for ax in axes:
    ax.axvspan(2000, 2019, color='0.92', zorder=0)
    ax.set_xlabel('Year')
    ax.set_ylabel('Runoff (Mt yr$^{-1}$)')
    ax.legend(fontsize=8)
axes[0].set_title('Total runoff (median, 5-95%)', loc='left')
axes[1].set_title('Components', loc='left')
plt.tight_layout()

- The runoff comes almost only from the glacier: the catchment is the glacier
  area of 2000, and the glacier changes little until then, so the ice-free
  part of the catchment (and its melt and rain) is small.
- The plausible runs agree on the observations, but not on the runoff: in
  2000-2019, the total runoff of the 5% and 95% runs differ by a factor of
  about 2.5. The melt on the glacier (snow and ice) is the largest component.

## Three runoff metrics

`runoff_metrics` computes, for each run, over 2000-2019 (the years of the
geodetic MB, grey above):

- `runoff`: the mean annual runoff (Mt yr⁻¹, i.e. million m³ of water per
  year);
- `runoff_monthly`: the mean runoff of each calendar month: the seasonal cycle;
- `summer_fraction`: the fraction of the annual runoff in July, August and
  September (one number for the seasonality); `peak_month`, the month with the
  largest runoff;
- `glacier_melt_share`: the melt on the glacier (snow and ice) divided by the
  runoff.

In [ ]:
m = runoff_metrics(dh)
metrics = ['runoff', 'summer_fraction', 'glacier_melt_share']
mlabels = {'runoff': 'mean runoff 2000-2019 (Mt yr$^{-1}$)',
           'summer_fraction': 'fraction of the runoff in Jul-Sep',
           'glacier_melt_share': 'share of glacier melt in the runoff'}

q = pd.DataFrame({v: weighted_quantile(m[v].values, wh, [0.05, 0.5, 0.95])
                  for v in metrics}, index=['q5', 'q50', 'q95']).T
q['spread (q95-q5) / q50'] = (q['q95'] - q['q5']) / q['q50']
display(q.round(3))
print('Peak month (weight of each month):')
pd.Series(wh, index=m.peak_month.values.astype(int)).groupby(level=0).sum().round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.5))
months = np.arange(1, 13)
clim = m.runoff_monthly.values
band(axes[0], months, clim, wh, C[0], 'median, 5-95%')
axes[0].set_xticks(months)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Runoff 2000-2019 (Mt month$^{-1}$)')
axes[0].legend(fontsize=8)
for ax, v in zip(axes[1:], metrics):
    ax.hist(m[v].values, bins=40, weights=wh, density=True, color=C[0], alpha=0.6)
    ax.set_xlabel(mlabels[v])
    ax.set_yticks([])
plt.tight_layout()

- **Mean runoff 2000-2019**: median 16.5 Mt yr⁻¹, 5-95% 12.0 to 31.7. The
  geodetic MB fixes the mass the glacier loses (913 kg m⁻² yr⁻¹ over 8.56 km²,
  about 7.8 Mt yr⁻¹, the same for all runs): the rest of the runoff is the
  precipitation on the catchment, which the observations hardly constrain.
- **Seasonality**: the runoff peaks in August in all runs, but the fraction of
  the runoff in July-September varies from 0.63 to 0.88.
- **Share of glacier melt** (snow and ice melt on the glacier): 0.58 to 0.93
  (median 0.81).

The histograms are spiky: about 165 runs carry most of the weight
(`neff`), and a run with a large weight makes a peak on its own. More
particles or several seeds would smooth them.

### Weighted runs, or a random draw?

We used all the runs which hold 99.9% of the weight, with their weights. The
other option of `select_particles` draws runs at random with the weights
(`n_resample`): fewer runs to rerun, all with the same weight, but the result
depends on the draw. What difference does it make here? We can find out
without rerunning anything: almost every draw falls among the runs we already
reran (a draw outside them has a probability of 0.1%). We compare the
quantiles of the metrics from the weighted runs with those of five draws of
500 runs (five different seeds):

In [ ]:
def quantiles(mm, weights):
    return {(v, q): weighted_quantile(mm[v].values, weights, [q])[0]
            for v in metrics for q in [0.05, 0.5, 0.95]}

rows = {'weighted runs': {**quantiles(m, wh), ('runs to rerun', ''): len(wh)}}
for seed in range(5):
    idx_d, w_d = select_particles(ds, n_resample=500, seed=seed)
    keep = np.isin(idx_d, m.particle.values)
    rows[f'500 draws, seed {seed}'] = {**quantiles(m.sel(particle=idx_d[keep]), w_d[keep]),
                                       ('runs to rerun', ''): len(idx_d)}
comp = pd.DataFrame(rows).T
comp.columns = pd.MultiIndex.from_tuples(comp.columns)
comp.round(3)

- The quantiles barely change between draws: by less than 0.7 Mt yr⁻¹ for the
  runoff (median and 95%: up to 0.6 and 0.4), by less than 0.03 for the
  fractions.
- Each draw needs about 240 reruns instead of 761.

Here, a draw of 500 is good enough. This is not guaranteed: when a few runs
carry a large part of the weight, the tails of the distributions move from one
draw to the next (in an earlier version of this notebook, the 95% quantile of
the runoff moved by 15% between draws). Several draws (seeds), or the weighted
runs, show how much the result can be trusted.

## Which parameters matter? A simple sensitivity analysis

Which parameters explain the spread of these metrics among the plausible runs?
We use three simple tools, on a sample of 2000 runs drawn with the weights
(so that each run of the sample counts equally):

- **scatter plots** of each metric against each parameter;
- the **Spearman rank correlation**: +1 or -1 if the metric always increases
  (or decreases) with the parameter, 0 if there is no monotonic relation;
- a **first-order index**: sort the runs by the value of one parameter, cut them
  into 10 groups of equal size, and compute the mean of the metric in each
  group. If the metric depended only on this parameter, these group means
  would vary as much as the metric itself; if it does not depend on it at all,
  they would all be the same. The index is the variance of the group means
  divided by the variance of the metric: the fraction of the spread which this
  parameter "explains" on its own (from 0 to 1).

**A limit, important here:** the plausible parameters are not independent.
The equifinality means that they are correlated (a high `melt_f` goes with a
high `prcp_fac`, a stiff ice with a warm initial state...). A metric can then
correlate with a parameter which does not act on it, only because this
parameter goes with another one which does. These indices describe the
plausible runs; they are not the Sobol indices of a variance-based sensitivity
analysis, which need independent samples of the parameters (see SALib, for
example).

In [ ]:
rng = np.random.default_rng(0)
draw = rng.choice(len(wh), 2000, p=wh / wh.sum())
sample = m.isel(particle=draw)
X = pd.DataFrame({p: sample[p].values for p in params})
Y = pd.DataFrame({v: sample[v].values for v in metrics})


def first_order_index(x, y, nbins=10):
    # Variance of the mean of y in nbins groups of equal size along x,
    # divided by the variance of y
    groups = pd.qcut(x, nbins, labels=False, duplicates='drop')
    means = pd.Series(y).groupby(groups).mean()
    sizes = pd.Series(y).groupby(groups).size()
    return np.average((means - y.mean()) ** 2, weights=sizes) / y.var(ddof=0)


spearman = pd.DataFrame({v: [X[p].corr(Y[v], method='spearman') for p in params]
                         for v in metrics}, index=params)
first_order = pd.DataFrame({v: [first_order_index(X[p].values, Y[v].values) for p in params]
                            for v in metrics}, index=params)
display(pd.concat({'Spearman': spearman, 'first-order index': first_order}, axis=1).round(2))

# How much the parameters are correlated among the plausible runs
X.corr(method='spearman').round(2)

In [ ]:
plabels = {'melt_f': 'melt_f', 'prcp_fac': 'prcp_fac', 'temp_bias': 'temp_bias (K)',
           'log10_glen_a_factor': 'log10 A factor',
           'initial_state_temp_offset': 'initial state offset (K)'}
fig, axes = plt.subplots(len(metrics), len(params), figsize=(17, 9), sharey='row')
for i, v in enumerate(metrics):
    for j, p in enumerate(params):
        ax = axes[i, j]
        ax.scatter(X[p], Y[v], s=2, color=C[0], alpha=0.3)
        if p in ('melt_f', 'prcp_fac'):
            ax.set_xscale('log')
        ax.text(0.03, 0.92, f'S1 = {first_order.loc[p, v]:.2f}', transform=ax.transAxes,
                fontsize=9)
        if i == len(metrics) - 1:
            ax.set_xlabel(plabels[p])
        if j == 0:
            ax.set_ylabel(mlabels[v], fontsize=8)
plt.tight_layout()

- **Mean runoff: `prcp_fac`** (rank correlation 0.99, first-order index 0.97).
  This is the mass balance at work: the mass loss is fixed by the geodetic
  MB, so more precipitation means more runoff. The spread of the runoff is the
  spread of `prcp_fac` which the observations allow (5-95%: 0.62 to 3.5), that
  is, mostly its prior (0.5 to 5). An observation of the precipitation, or of
  the winter MB (first notebook), would constrain it. `temp_bias` (index 0.54)
  probably matters only through its correlation with `prcp_fac` among the
  plausible runs (0.58, table above).
- **Seasonality: `temp_bias`** (index 0.98, an almost perfect line): with a
  warmer climate (higher `temp_bias`), more of the precipitation falls as
  rain, and the melt season is probably longer, which spreads the runoff over
  more months (not checked month by month). `melt_f` (0.85) is strongly
  correlated with `temp_bias` (-0.92).
- **Share of glacier melt: `temp_bias` and `prcp_fac`** (indices 0.88 and
  0.76): a warm, wet glacier gets more rain; a cold, dry one gets its runoff
  from melt.
- **Glen's A and the initial state offset** matter little for these 2000-2019
  metrics (indices 0.05 to 0.17, no clear pattern in the scatter plots). They
  would matter more for metrics which depend on the volume of the glacier,
  such as its future runoff: the plausible glaciers of 2020 differ by a factor
  of 3 in volume (previous notebook).

The sum of the first-order indices of one metric is larger than 1 (for
example 1.8 for the mean runoff): the parameters are correlated, and the
indices count the same spread several times. This is the limit described
above.

# Summary

- The plausible runs of Vernagtferner, which all reproduce the geodetic MB,
  the RGI area and the IceBoost thickness, disagree on the runoff of the same
  years: a factor of about 2.5 in the mean annual runoff 2000-2019, from 0.63
  to 0.88 for the fraction of the runoff in July-September, from 0.58 to 0.93
  for the share of glacier melt.
- On these metrics, the MB parameters matter, not Glen's A or the initial
  state: `prcp_fac` sets the total runoff (the geodetic MB fixes the mass
  loss, the precipitation does the rest), `temp_bias` the seasonality, both
  the share of glacier melt.
- The rerun of the plausible runs with hydrological output (761 runs, 73 s
  on 32 processes of our cluster) is much cheaper than computing it in every
  AdaPBS run. A random draw of 500 runs needs about a third of the reruns,
  and is good enough here.
- The spread of the runoff follows from the prior bounds of the parameters
  which the observations do not constrain (here `prcp_fac`): these bounds
  matter, and should be discussed.

Open questions and next steps:

- Other metrics: changes of runoff (e.g. peak water), projections with the
  plausible runs, the runoff of other periods.
- A variance-based sensitivity analysis (SALib) needs independent parameter
  samples: either on the prior (all runs, not only the plausible ones), or
  with a method which accounts for the correlations.
- Other glaciers, and the choices of the previous notebook (errors, initial
  state, spinup length).